# 03 · EDA — Peras / PostHog

**Purpose:** Turn events_clean.parquet into the acquisition / activation /
engagement / retention / experiments analysis, and produce the **public-safe
aggregated tier** from it.

**Safety contract for this notebook**
- Reads data/raw/events_clean.parquet (private tier, gitignored). No cell in this
  notebook prints a raw person-level row -- only aggregated/statistical summaries,
  because notebook *outputs* get committed even though the data files don't.
- Everything written to data/aggregated/ has been aggregated to group level (no
  person_id/distinct_id/account_id column survives) and small-cell suppressed
  (SMALL_CELL_THRESHOLD -- any group, or any count within a group, below it is
  dropped, not just masked).
- data/aggregated/ being committable doesn't mean it's cleared to publish --
  CLAUDE.md still requires written clearance from Peras before anything goes
  public.
- This is dev-scale data (dozens of people, not production volume), so most
  breakdowns below will get suppressed by the small-cell rule. That's the rule
  working as intended -- expect real signal once this runs against production data.

**Before running:** run 01_extract.ipynb then 02_clean_audit.ipynb first so
data/raw/events_clean.parquet exists.


In [1]:
from pathlib import Path

import pandas as pd

CLEAN_DIR = Path("../data/raw")
AGG_DIR = Path("../data/aggregated")
IN_PATH = CLEAN_DIR / "events_clean.parquet"

assert IN_PATH.exists(), f"{IN_PATH} not found -- run 01_extract.ipynb and 02_clean_audit.ipynb first."

events = pd.read_parquet(IN_PATH)
events["timestamp"] = pd.to_datetime(events["timestamp"], utc=True)

n_people = events["person_id"].nunique()
first_seen = events.groupby("person_id")["timestamp"].min()

print(f"Loaded {len(events):,} rows, {n_people} unique people, "
      f"{events['event'].nunique()} event types.")

SMALL_CELL_THRESHOLD = 5  # CLAUDE.md: k-anonymity floor for anything leaving the private tier


def suppress_small_cells(df, count_cols, threshold=SMALL_CELL_THRESHOLD):
    """Drop any row where any of count_cols is below threshold (CLAUDE.md:
    small-cell suppression -- groups under ~5 people never leave this notebook).
    Checks every count in the row, not just the group denominator -- a rare
    numerator (e.g. "1 person activated") is just as identifying as a small group."""
    if isinstance(count_cols, str):
        count_cols = [count_cols]
    small = pd.Series(False, index=df.index)
    for c in count_cols:
        small = small | (df[c] < threshold)
    n_dropped = int(small.sum())
    if n_dropped:
        print(f"  suppressed {n_dropped} row(s) with a count below {threshold} in {count_cols}")
    return df.loc[~small].reset_index(drop=True)


def first_touch(df, col):
    """First non-null value of col per person, ordered by time -- used for
    first-touch channel/experiment-variant attribution."""
    d = df.dropna(subset=[col]).sort_values("timestamp")
    return d.groupby("person_id")[col].first()

Loaded 101,224 rows, 3488 unique people, 139 event types.


## Acquisition — where do users come from, and which experiments bring quality traffic?

First-touch attribution per person: earliest non-null utm_source, falling back to
referring_domain, falling back to "direct/unknown". Then activation rate by channel,
so "where traffic comes from" is judged by whether it converts, not just volume.


In [2]:
first_utm_source = first_touch(events, "utm_source")
first_referring_domain = first_touch(events, "referring_domain")

acquisition = pd.DataFrame(index=events["person_id"].unique())
acquisition.index.name = "person_id"
acquisition["channel"] = first_utm_source
acquisition["channel"] = acquisition["channel"].fillna(first_referring_domain)
acquisition["channel"] = acquisition["channel"].fillna("direct/unknown")
acquisition = acquisition.reset_index()

acquisition["channel"].value_counts()


channel
x                                          2951
$direct                                     175
t.co                                        125
direct/unknown                               97
www.google.com                               56
google                                       41
reddit                                       19
accounts.google.com                           8
peras.app                                     3
duckduckgo.com                                2
checkout.stripe.com                           2
internal_test                                 1
search.brave.com                              1
www.bing.com                                  1
com.google.android.googlequicksearchbox       1
temp-mail.org                                 1
org.telegram.plus                             1
www.reddit.com                                1
www.forelook.com                              1
www.yamli.com                                 1
Name: count, dtype: int64

In [3]:
SIGNUP_EVENTS = ["account.signed_up"]
ACTIVATION_EVENTS = ["generation.completed", "generation.first_completed"]

signed_up_people = set(events.loc[events["event"].isin(SIGNUP_EVENTS), "person_id"])
activated_people = set(events.loc[events["event"].isin(ACTIVATION_EVENTS), "person_id"])

acquisition["signed_up"] = acquisition["person_id"].isin(signed_up_people)
acquisition["activated"] = acquisition["person_id"].isin(activated_people)

by_channel = (
    acquisition.groupby("channel")
    .agg(
        people=("person_id", "nunique"),
        signed_up=("signed_up", "sum"),
        activated=("activated", "sum"),
    )
    .reset_index()
)
by_channel["signup_rate"] = (by_channel["signed_up"] / by_channel["people"]).round(3)
by_channel["activation_rate"] = (by_channel["activated"] / by_channel["people"]).round(3)

by_channel_safe = suppress_small_cells(by_channel, ["people", "signed_up", "activated"])
by_channel_safe

  suppressed 20 row(s) with a count below 5 in ['people', 'signed_up', 'activated']


,channel,people,signed_up,activated,signup_rate,activation_rate


## Signup — how many acquired people create an account?

Distinct from activation (first textbook generated): signup is `account.signed_up`,
the gate into having an account at all. Kept as its own stage rather than folded
into "activated" so a signup/activation gap (people who make an account but never
generate anything) is visible instead of being hidden inside one metric.


In [ ]:
n_signed_up = len(signed_up_people)
if n_signed_up == 0 or n_signed_up >= SMALL_CELL_THRESHOLD:
    print(f"Signed up: {n_signed_up} of {n_people} people ({n_signed_up / n_people:.1%})")
else:
    print(f"Fewer than {SMALL_CELL_THRESHOLD} signed-up people in this dev-scale run -- suppressing the signup count and rate.")

if n_signed_up >= SMALL_CELL_THRESHOLD:
    signed_up_at = (
        events.loc[events["event"].isin(SIGNUP_EVENTS)]
        .groupby("person_id")["timestamp"].min()
    )
    time_to_signup_hours = (signed_up_at - first_seen.loc[signed_up_at.index]).dt.total_seconds() / 3600
    display(time_to_signup_hours.describe())

    n_signed_up_and_activated = len(signed_up_people & activated_people)
    if n_signed_up_and_activated == 0 or n_signed_up_and_activated >= SMALL_CELL_THRESHOLD:
        signup_to_activation_rate = n_signed_up_and_activated / n_signed_up
        print(f"Of signed-up people, {signup_to_activation_rate:.1%} went on to activate.")
    else:
        print(f"Fewer than {SMALL_CELL_THRESHOLD} signed-up people also activated -- suppressing the signup-to-activation rate.")
else:
    print(f"Fewer than {SMALL_CELL_THRESHOLD} signed-up people in this dev-scale run -- suppressing the time-to-signup distribution and the signup-to-activation rate.")
    signed_up_at = None
    time_to_signup_hours = None

## Experiments — do the tested landing / campaign / consent variants move activation?

Peras runs three separate experiments as real properties (confirmed via PostHog's
schema, not guessed): landing_variant, campaign_variant, sample_consent_variant.
Same first-touch approach: each person's first-assigned variant, joined to whether
they later activated.


In [5]:
EXPERIMENT_COLS = ["landing_variant", "campaign_variant", "sample_consent_variant"]

experiment_results = {}
for col in EXPERIMENT_COLS:
    variant_by_person = first_touch(events, col)
    tmp = acquisition.set_index("person_id").copy()
    tmp["variant"] = variant_by_person
    tmp = tmp.dropna(subset=["variant"])

    summary = (
        tmp.groupby("variant")
        .agg(
            people=("activated", "size"),
            signed_up=("signed_up", "sum"),
            activated=("activated", "sum"),
        )
        .reset_index()
    )
    summary["signup_rate"] = (summary["signed_up"] / summary["people"]).round(3)
    summary["activation_rate"] = (summary["activated"] / summary["people"]).round(3)
    experiment_results[col] = suppress_small_cells(summary, ["people", "signed_up", "activated"])

    print(f"--- {col} ---")
    display(experiment_results[col])

  suppressed 14 row(s) with a count below 5 in ['people', 'signed_up', 'activated']
--- landing_variant ---


,variant,people,signed_up,activated,signup_rate,activation_rate


  suppressed 3 row(s) with a count below 5 in ['people', 'signed_up', 'activated']
--- campaign_variant ---


,variant,people,signed_up,activated,signup_rate,activation_rate


  suppressed 2 row(s) with a count below 5 in ['people', 'signed_up', 'activated']
--- sample_consent_variant ---


,variant,people,signed_up,activated,signup_rate,activation_rate


## Activation — what fraction generate their first textbook, and how fast?

Activation = first generation.completed / generation.first_completed event.
"How fast" = time from first-seen to that event, for people who activated.


In [ ]:
activated_at = (
    events.loc[events["event"].isin(ACTIVATION_EVENTS)]
    .groupby("person_id")["timestamp"].min()
)

n_activated = len(activated_at)
if n_activated == 0 or n_activated >= SMALL_CELL_THRESHOLD:
    print(f"Activated: {n_activated} of {n_people} people ({n_activated / n_people:.1%})")
else:
    print(f"Fewer than {SMALL_CELL_THRESHOLD} activated people in this dev-scale run -- suppressing the activation count and rate.")

if n_activated >= SMALL_CELL_THRESHOLD:
    time_to_activate_hours = (activated_at - first_seen.loc[activated_at.index]).dt.total_seconds() / 3600
    display(time_to_activate_hours.describe())
else:
    print(f"Fewer than {SMALL_CELL_THRESHOLD} activated people in this dev-scale run -- suppressing the time-to-activate distribution.")
    time_to_activate_hours = None

## Engagement — how deeply do activated users use the product?

Projects (textbooks) per person, distinct generation/reader modes touched, reader
depth, and sessions -- restricted to people who activated, since "depth of use"
only means something once someone has gotten past first activation.


In [ ]:
engagement = (
    events.groupby("person_id")
    .agg(
        projects=("project_id", "nunique"),
        generation_modes=("generation_mode", lambda s: s.dropna().nunique()),
        reader_modes=("reader_mode", lambda s: s.dropna().nunique()),
        max_reader_sections=("reader_written_sections", "max"),
        sessions=("session_id", "nunique"),
    )
)

engaged = engagement.loc[engagement.index.isin(activated_people)]

if len(engaged) >= SMALL_CELL_THRESHOLD:
    engagement_summary = engaged.describe()
    display(engagement_summary)
else:
    print(f"Fewer than {SMALL_CELL_THRESHOLD} activated people in this dev-scale run -- suppressing the distribution.")
    engagement_summary = pd.DataFrame()

## Retention — do they come back, and which cohorts stick?

Cohort = calendar week of first activity. Retention = fraction of each cohort still
active N weeks later. Aggregation in DuckDB (per the project's toolchain -- local
SQL over the extracts for the heavier group-by work), never materializing a
person-level table for this step.


In [8]:
import duckdb

con = duckdb.connect()
retention_pairs = con.execute("""
    WITH cohort AS (
        SELECT person_id, date_trunc('week', min(timestamp)) AS cohort_week
        FROM events
        GROUP BY person_id
    ),
    activity AS (
        SELECT DISTINCT person_id, date_trunc('week', timestamp) AS activity_week
        FROM events
    )
    SELECT
        c.cohort_week,
        datediff('week', c.cohort_week, a.activity_week) AS weeks_since_cohort,
        count(DISTINCT a.person_id) AS active_people
    FROM activity a
    JOIN cohort c USING (person_id)
    GROUP BY c.cohort_week, weeks_since_cohort
    ORDER BY c.cohort_week, weeks_since_cohort
""").df()

cohort_sizes = con.execute("""
    WITH cohort AS (
        SELECT person_id, date_trunc('week', min(timestamp)) AS cohort_week
        FROM events
        GROUP BY person_id
    )
    SELECT cohort_week, count(DISTINCT person_id) AS cohort_size
    FROM cohort
    GROUP BY cohort_week
""").df()

retention = retention_pairs.merge(cohort_sizes, on="cohort_week")
retention["retention_rate"] = (retention["active_people"] / retention["cohort_size"]).round(3)

retention_safe = suppress_small_cells(retention, ["cohort_size", "active_people"])
retention_safe

  suppressed 5 row(s) with a count below 5 in ['cohort_size', 'active_people']


,cohort_week,weeks_since_cohort,active_people,cohort_size,retention_rate
0,2026-06-15 00:00:00-05:00,0,50,50,1.000
1,2026-06-22 00:00:00-05:00,0,628,628,1.000
2,2026-06-22 00:00:00-05:00,1,37,628,0.059
3,2026-06-22 00:00:00-05:00,2,11,628,0.018
4,2026-06-29 00:00:00-05:00,0,1406,1406,1.000
5,2026-06-29 00:00:00-05:00,1,68,1406,0.048
6,2026-06-29 00:00:00-05:00,2,17,1406,0.012
7,2026-07-06 00:00:00-05:00,0,1079,1079,1.000
8,2026-07-06 00:00:00-05:00,1,28,1079,0.026
9,2026-07-13 00:00:00-05:00,0,325,325,1.000


## Putting it together — funnel summary

One table spanning all four stages, so the whole funnel is visible at a glance.
"Engaged" and "retained" thresholds are tunable constants, not fixed rules.


In [9]:
ENGAGED_MIN_SESSIONS = 2  # tunable: what counts as "engaged" beyond just activating

sessions_per_person = events.groupby("person_id")["session_id"].nunique()
weeks_active_per_person = events.groupby("person_id")["timestamp"].apply(lambda s: s.dt.to_period("W").nunique())

funnel_summary = pd.DataFrame([
    {"stage": "acquired", "people": n_people},
    {"stage": "signed_up", "people": len(signed_up_people)},
    {"stage": "activated", "people": len(activated_at)},
    {"stage": f"engaged ({ENGAGED_MIN_SESSIONS}+ sessions)", "people": int((sessions_per_person >= ENGAGED_MIN_SESSIONS).sum())},
    {"stage": "retained (active in 2+ distinct weeks)", "people": int((weeks_active_per_person > 1).sum())},
])
funnel_summary["pct_of_acquired"] = (funnel_summary["people"] / n_people).round(3)

funnel_summary_safe = suppress_small_cells(funnel_summary, ["people"])
funnel_summary_safe

C:\Users\Faith\AppData\Local\Temp\ipykernel_24896\2462513776.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  weeks_active_per_person = events.groupby("person_id")["timestamp"].apply(lambda s: s.dt.to_period("W").nunique())


  suppressed 1 row(s) with a count below 5 in ['people']


,stage,people,pct_of_acquired
0,acquired,3488,1.000
1,signed_up,116,0.033
2,engaged (2+ sessions),821,0.235
3,retained (active in 2+ distinct weeks),154,0.044


## Write aggregated, public-safe outputs

Every table below has already passed through suppress_small_cells -- nothing with
a group or count under SMALL_CELL_THRESHOLD survives to this point.


In [10]:
AGG_DIR.mkdir(parents=True, exist_ok=True)

by_channel_safe.to_csv(AGG_DIR / "acquisition_by_channel.csv", index=False)

for col, df in experiment_results.items():
    if not df.empty:
        df.to_csv(AGG_DIR / f"experiment_{col}.csv", index=False)

if not engagement_summary.empty:
    engagement_summary.reset_index().to_csv(AGG_DIR / "engagement_summary.csv", index=False)

retention_safe.to_csv(AGG_DIR / "retention_by_cohort_week.csv", index=False)
funnel_summary_safe.to_csv(AGG_DIR / "funnel_summary.csv", index=False)

print("Wrote aggregated, public-safe outputs to", AGG_DIR.resolve())
for f in sorted(AGG_DIR.glob("*.csv")):
    print(" -", f.name)

Wrote aggregated, public-safe outputs to C:\Users\Faith\Desktop\peras-analytics\peras-analytics\data\aggregated
 - acquisition_by_channel.csv
 - funnel_summary.csv
 - retention_by_cohort_week.csv


### Bonus: one combined CSV for quick visualization

All the safe/suppressed tables above, stacked into one tidy (analysis, dimension,
metric, value) file instead of five separate ones -- easier to drop straight into a
chart or pivot table. Same suppression already applied to every table going in, so
this is exactly as safe as the individual CSVs -- still not cleared to publish
without Peras's sign-off, per CLAUDE.md.


In [11]:
def to_long(df, analysis_name, dimension_cols):
    """Melt a wide safe/summary table into tidy (analysis, dimension, metric, value)
    rows so tables with completely different shapes can stack into one file."""
    d = df.copy()
    if len(dimension_cols) == 1:
        dimension = d[dimension_cols[0]].astype(str)
    else:
        dimension = d[dimension_cols].astype(str).agg(" / ".join, axis=1)
    value_cols = [c for c in d.columns if c not in dimension_cols]
    out = d[value_cols].copy()
    out.insert(0, "dimension", dimension)
    long_df = out.melt(id_vars="dimension", var_name="metric", value_name="value")
    long_df.insert(0, "analysis", analysis_name)
    return long_df


long_frames = [to_long(by_channel_safe, "acquisition_by_channel", ["channel"])]

for col, df in experiment_results.items():
    if not df.empty:
        long_frames.append(to_long(df, f"experiment_{col}", ["variant"]))

long_frames.append(to_long(retention_safe, "retention", ["cohort_week", "weeks_since_cohort"]))
long_frames.append(to_long(funnel_summary_safe, "funnel_summary", ["stage"]))

if not engagement_summary.empty:
    eng = engagement_summary.reset_index().rename(columns={"index": "stat"})
    long_frames.append(to_long(eng, "engagement_summary", ["stat"]))

combined_long = pd.concat(long_frames, ignore_index=True)

combined_path = AGG_DIR / "eda_combined_long.csv"
combined_long.to_csv(combined_path, index=False)
print(f"Wrote {len(combined_long):,} rows -> {combined_path.resolve()}")
combined_long.head(10)


Wrote 38 rows -> C:\Users\Faith\Desktop\peras-analytics\peras-analytics\data\aggregated\eda_combined_long.csv


,analysis,dimension,metric,value
0,retention,2026-06-15 00:00:00-05:00 / 0,active_people,50.0
1,retention,2026-06-22 00:00:00-05:00 / 0,active_people,628.0
2,retention,2026-06-22 00:00:00-05:00 / 1,active_people,37.0
3,retention,2026-06-22 00:00:00-05:00 / 2,active_people,11.0
4,retention,2026-06-29 00:00:00-05:00 / 0,active_people,1406.0
5,retention,2026-06-29 00:00:00-05:00 / 1,active_people,68.0
6,retention,2026-06-29 00:00:00-05:00 / 2,active_people,17.0
7,retention,2026-07-06 00:00:00-05:00 / 0,active_people,1079.0
8,retention,2026-07-06 00:00:00-05:00 / 1,active_people,28.0
9,retention,2026-07-13 00:00:00-05:00 / 0,active_people,325.0


---

## Findings so far (dev-scale data -- re-run at production volume before trusting these)

- **Acquisition:** channel-level activation rates are in `acquisition_by_channel.csv`
  once enough people exist per channel to clear suppression. So what: once populated,
  reallocate acquisition spend toward whichever channel shows both real volume and a
  competitive activation rate -- volume alone isn't quality.
- **Experiments:** `experiment_*.csv` show activation rate by variant for each of the
  three tests Peras runs. So what: only trust a variant delta once every arm clears
  the small-cell floor -- with true production volume, this becomes the actual
  ship/no-ship call for each experiment.
- **Activation & engagement:** time-to-activate and per-person usage depth are in the
  notebook output above (and `engagement_summary.csv` if populated). So what: a long
  tail on time-to-activate points at onboarding friction worth shortening.
- **Retention:** `retention_by_cohort_week.csv` is the raw material for a standard
  retention curve. So what: compare cohorts against each other (e.g. pre/post an
  onboarding change) once volume is real, rather than reading one curve in isolation.

## Next

Tableau dashboards on `data/aggregated/` (`tableau/`), then the findings write-up
(`reports/findings_memo.md`) -- both still gated on written clearance from Peras
before anything goes public, per CLAUDE.md.
